In [1]:
print("Feast Feature Store")

Feast Feature Store


In [9]:
import pandas as pd

df = pd.read_csv("../../data/Housing_processed.csv")
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,1.046726,1.403419,1.421812,1.378217,1,0,0,0,1,1.517692,1,0
1,12250000,1.757010,1.403419,5.405809,2.532024,1,0,0,0,1,2.679409,0,0
2,12250000,2.218232,0.047278,1.421812,0.224410,1,0,1,0,0,1.517692,1,1
3,12215000,1.083624,1.403419,1.421812,0.224410,1,0,1,0,1,2.679409,1,0
4,11410000,1.046726,1.403419,-0.570187,0.224410,1,1,1,0,1,1.517692,0,0


In [10]:
timestamps = pd.date_range(
    end=pd.Timestamp.now(),
    start=pd.Timestamp.now(),
    periods=len(df),
    freq=None
).to_frame(name="event_timestamp", index=False)

df['event_timestamp'] = timestamps['event_timestamp']
df['house_id'] = range(1, len(df) + 1)
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus,event_timestamp,house_id
0,13300000,1.046726,1.403419,1.421812,1.378217,1,0,0,0,1,1.517692,1,0,2025-12-17 14:24:03.120401000,1
1,12250000,1.757010,1.403419,5.405809,2.532024,1,0,0,0,1,2.679409,0,0,2025-12-17 14:24:03.120398750,2
2,12250000,2.218232,0.047278,1.421812,0.224410,1,0,1,0,0,1.517692,1,1,2025-12-17 14:24:03.120396500,3
3,12215000,1.083624,1.403419,1.421812,0.224410,1,0,1,0,1,2.679409,1,0,2025-12-17 14:24:03.120394250,4
4,11410000,1.046726,1.403419,-0.570187,0.224410,1,1,1,0,1,1.517692,0,0,2025-12-17 14:24:03.120392000,5


In [11]:
# Splitting the dataset into features (X) and target (y)
X = df.drop(columns=['price'])
y = pd.DataFrame(df[['house_id', 'price', 'event_timestamp']])

print("X features:")
print(X.head())

print("y target:")
print(y.head())

X features:
       area  bedrooms  bathrooms   stories  mainroad  guestroom  basement  \
0  1.046726  1.403419   1.421812  1.378217         1          0         0   
1  1.757010  1.403419   5.405809  2.532024         1          0         0   
2  2.218232  0.047278   1.421812  0.224410         1          0         1   
3  1.083624  1.403419   1.421812  0.224410         1          0         1   
4  1.046726  1.403419  -0.570187  0.224410         1          1         1   

   hotwaterheating  airconditioning   parking  prefarea  furnishingstatus  \
0                0                1  1.517692         1                 0   
1                0                1  2.679409         0                 0   
2                0                0  1.517692         1                 1   
3                0                1  2.679409         1                 0   
4                0                1  1.517692         0                 0   

                event_timestamp  house_id  
0 2025-12-17 14:24

In [7]:
import sqlalchemy as db
from sqlalchemy import text
from dotenv import load_dotenv
import os

load_dotenv()  # Load environment variables from .env file

connstr = os.getenv("POSTGRE_SQL_CONN_STR")
offline_db = os.getenv("POSTGRE_SQL_OFFLINE_DB")
full_connstr = f"{connstr}/{offline_db}"

engine = db.create_engine(full_connstr)


with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user"))
    print(result.fetchone())

('feast_offline', 'postgres')


In [ ]:
X.to_sql('house_features_sql', engine, if_exists='replace', index=False)
y.to_sql('house_target_sql', engine, if_exists='replace', index=False)

-1

In [19]:
parent_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
X.to_parquet(path=os.path.join(parent_dir, 'house_features.parquet'), index=False)
y.to_parquet(path=os.path.join(parent_dir, 'house_target.parquet'), index=False)